## Análisis Exploratorio de Datos (EDA)

### Objetivo
Este notebook realiza un **análisis exploratorio** sobre los datasets extraídos del INE para verificar la integridad, calidad y estructura de los datos antes de las transformaciones.

Se examinan los siete DataFrames generados en la fase de extracción:
- **Empresas constituidas** (`empresas_constituidas.csv`) — Verificación de tipos societarios, consistencia de capital y nº de sociedades.
- **Empresas disueltas** (`empresas_disueltas.csv`) — Distribución por causa de disolución y territorio.
- **IPC** (`ipc.csv`) — Rango de valores, detección de outliers y cardinalidad de dimensiones.
- **Tablas dimensionales** (`sectores_ipc`, `territorio`, `tiempo`, `tipo_medida`) — Validación de claves y completitud.

### Metodología
1. **Carga** de los CSV desde `../files/data_raw/`.
2. **Inspección** mediante `info()`, `describe()`, `sample()` y `value_counts()`.
3. **Detección de inconsistencias** — Identificación de filas duplicadas (como los agregados "Mercantiles" que sumarizan a S.A. y S.L.) y valores anómalos.
4. **Documentación de hallazgos** — Conclusiones que guiarán las transformaciones en la siguiente etapa del pipeline.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración de el estilo de los graficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)

1. Revisamos los archivos exportados para comprobar la integridad de los datos

In [2]:
df_empr_const = pd.read_csv('../files/data_raw/empresas_constituidas.csv', index_col=0)

In [3]:
df_empr_const.sample(5)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
1184,Cantabria,201905,Mercantiles,50,318000
1208,Cantabria,201705,Mercantiles,80,626000
7376,"Navarra, Comunidad Foral de",201609,Sociedades anónimas,1,60000
4990,"Balears, Illes",201311,Sociedades anónimas,0,0
1594,Castilla - La Mancha,202111,Mercantiles,232,9553000


In [4]:
df_empr_const.info()

<class 'pandas.DataFrame'>
RangeIndex: 16720 entries, 1 to 16720
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   territorio         16720 non-null  str  
 1   id_tiempo          16720 non-null  int64
 2   tipo               16720 non-null  str  
 3   numero_sociedades  16720 non-null  int64
 4   capital            16720 non-null  int64
dtypes: int64(3), str(2)
memory usage: 653.3 KB


In [5]:
df_empr_const.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,16720.0,2.016737e+05,5.294115e+02,200801.0,201207.75,201702.5,202109.25,2.026040e+05
numero_sociedades,16720.0,2.134034e+02,4.471220e+02,0.0,0.00,7.0,215.00,3.173000e+03
capital,16720.0,1.544442e+07,1.489657e+08,0.0,0.00,442000.0,7895000.00,1.259172e+10


In [6]:
df_empr_const.describe(include='string').T

,count,unique,top,freq
territorio,16720,19,Andalucía,880
tipo,16720,4,Mercantiles,4180


In [7]:
df_empr_const["territorio"].unique()

<StringArray>
[                  'Andalucía',                      'Aragón',
     'Asturias, Principado de',              'Balears, Illes',
                    'Canarias',                   'Cantabria',
             'Castilla y León',        'Castilla - La Mancha',
                    'Cataluña',        'Comunitat Valenciana',
                 'Extremadura',                     'Galicia',
        'Madrid, Comunidad de',           'Murcia, Región de',
 'Navarra, Comunidad Foral de',                  'País Vasco',
                   'Rioja, La',                       'Ceuta',
                     'Melilla']
Length: 19, dtype: str

Pendiente de arreglar los nombres, minúsculas, tildes, etc. normalizar la columna

In [8]:
df_empr_const["tipo"].unique()

<StringArray>
[                           'Mercantiles',
                    'Sociedades anónimas',
 'Sociedades de responsabilidad limitada',
       'S. Comanditarias y S. Colectivas']
Length: 4, dtype: str

Tras revisar los tipos de empresa, nos damos cuenta que "Mercantiles" es un total de las empresas SL, y SA, vamos a comprobarlo.

In [9]:
df_empr_const[df_empr_const["tipo"] == "Mercantiles"].sample(10)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
1271,Cantabria,201202,Mercantiles,67,3530000
1030,Canarias,201311,Mercantiles,228,10722000
355,Aragón,201502,Mercantiles,179,2953000
2869,"Murcia, Región de",202508,Mercantiles,139,9990000
4025,Melilla,202012,Mercantiles,8,172000
275,Aragón,202110,Mercantiles,132,3733000
1733,Castilla - La Mancha,201004,Mercantiles,243,49150000
1223,Cantabria,201602,Mercantiles,96,661000
947,Canarias,202010,Mercantiles,226,3547000


In [10]:
df_empr_const["tipo"].value_counts()

tipo
Mercantiles                               4180
Sociedades anónimas                       4180
Sociedades de responsabilidad limitada    4180
S. Comanditarias y S. Colectivas          4180
Name: count, dtype: int64

La razón que los valores totales coinciden es porque hay una fila por fecha, independientemente si hay disueltas o no (0)

In [11]:
df_empr_const[df_empr_const["tipo"] == "Mercantiles"].shape

(4180, 5)

In [12]:
df_empr_const[(df_empr_const["territorio"] == "Melilla") & (df_empr_const["id_tiempo"] == 202505)].head(100)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
3972,Melilla,202505,Mercantiles,10,1037000
8152,Melilla,202505,Sociedades anónimas,0,0
12332,Melilla,202505,Sociedades de responsabilidad limitada,10,1037000
16512,Melilla,202505,S. Comanditarias y S. Colectivas,0,0


In [13]:
df_empr_const[(df_empr_const["territorio"] == "Canarias") & (df_empr_const["id_tiempo"] == 202006)].head(100)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
951,Canarias,202006,Mercantiles,186,155088000
5131,Canarias,202006,Sociedades anónimas,0,0
9311,Canarias,202006,Sociedades de responsabilidad limitada,186,155088000
13491,Canarias,202006,S. Comanditarias y S. Colectivas,0,0


In [14]:
df_empr_const[(df_empr_const["territorio"] == "Andalucía") & (df_empr_const["id_tiempo"] == 202407)].head(100)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
22,Andalucía,202407,Mercantiles,1564,78298000
4202,Andalucía,202407,Sociedades anónimas,2,1840000
8382,Andalucía,202407,Sociedades de responsabilidad limitada,1562,76458000
12562,Andalucía,202407,S. Comanditarias y S. Colectivas,0,0


In [15]:
df_empr_const.columns

Index(['territorio', 'id_tiempo', 'tipo', 'numero_sociedades', 'capital'], dtype='str')

In [16]:
df_empr_const.duplicated().sum()

np.int64(0)

Confirmamos nuestras sospechas, procederemos en el paso de transformación a eliminar esas filas. df_empr_const["tipo"] == "Mercantiles"

------------------------------------

In [17]:
df_empr_dis = pd.read_csv('../files/data_raw/empresas_disueltas.csv', index_col=0)

In [18]:
df_empr_dis.sample(5)

,territorio,id_tiempo,razon,numero_sociedades
id_dis,,,,
5599,Castilla y León,201802,Por fusión,6
213,Andalucía,200808,Voluntaria,105
4978,"Balears, Illes",201411,Por fusión,6
1229,Cantabria,201508,Voluntaria,10
1119,Cantabria,202410,Voluntaria,19


In [19]:
df_empr_dis.info()

<class 'pandas.DataFrame'>
RangeIndex: 12540 entries, 1 to 12540
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   territorio         12540 non-null  str  
 1   id_tiempo          12540 non-null  int64
 2   razon              12540 non-null  str  
 3   numero_sociedades  12540 non-null  int64
dtypes: int64(2), str(2)
memory usage: 392.0 KB


In [20]:
df_empr_dis.shape

(12540, 4)

In [21]:
df_empr_dis.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,12540.0,201673.700000,529.416812,200801.0,201207.75,201702.5,202109.25,202604.0
numero_sociedades,12540.0,32.513238,71.914047,0.0,2.00,8.0,31.00,1114.0


In [22]:
df_empr_dis.describe(include='string').T

,count,unique,top,freq
territorio,12540,19,Andalucía,660
razon,12540,3,Voluntaria,4180


In [23]:
df_empr_dis["razon"].value_counts()

razon
Voluntaria    4180
Por fusión    4180
Otras         4180
Name: count, dtype: int64

In [24]:
df_empr_dis.duplicated().sum()

np.int64(0)

La razón que los valores totales coinciden es porque hay una fila por fecha, independientemente si hay disueltas o no (0)

-------------------------

In [25]:
df_ipc = pd.read_csv('../files/data_raw/ipc.csv')

In [26]:
df_ipc.sample(10)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
298116,201504,19,3,2,0.000
2644,202512,1,3,2,0.700
269698,201501,17,7,1,88.499
241528,201807,15,11,1,89.531
197647,201210,13,1,3,3.300
269787,200708,17,7,1,80.598
327989,201605,20,14,4,1.400
176209,201611,11,11,2,0.100
210642,200403,13,12,3,4.000
133030,202511,9,2,3,2.200


In [27]:
df_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 328162 entries, 0 to 328161
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id_tiempo      328162 non-null  int64  
 1   id_territorio  328162 non-null  int64  
 2   id_sector      328162 non-null  int64  
 3   id_medida      328162 non-null  int64  
 4   valor_ipc      328162 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 12.5 MB


In [28]:
df_ipc.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,328162.0,201377.778817,705.032930,200201.0,200802.0,201403.0,202004.0000,202606.000
id_territorio,328162.0,10.499942,5.766320,1.0,5.0,10.0,15.0000,20.000
id_sector,328162.0,7.499960,4.031155,1.0,4.0,7.0,11.0000,14.000
id_medida,328162.0,2.500000,1.118033,1.0,2.0,2.5,3.0000,4.000
valor_ipc,328162.0,21.500742,37.058520,-22.4,0.1,1.5,37.3585,258.216


In [29]:
df_ipc[df_ipc["valor_ipc"] == -22.4].head()

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
185797,202308,12,5,3,-22.4


In [30]:
df_ipc[df_ipc["valor_ipc"] <0].sample(10)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
238378,201206,15,8,2,-1.7
90605,202011,6,8,2,-1.1
82751,201602,6,1,3,-0.8
73131,201201,5,7,2,-0.4
184476,201108,12,4,2,-0.7
73673,201509,5,7,4,-1.0
37325,201701,3,4,4,-14.5
237162,201602,15,7,2,-0.2
89867,200902,6,7,3,-0.8
185558,201902,12,5,2,-1.7


In [31]:
df_ipc.duplicated().sum()

np.int64(0)

----------------

In [32]:
df_sectores_ipc = pd.read_csv('../files/data_raw/sectores_ipc.csv')

In [33]:
df_sectores_ipc.sample(10)

,id_sector,nombre_sector
12,13,Seguros y servicios financieros
1,2,Alimentos y bebidas no alcohólicas
7,8,Transporte
10,11,Enseñanza
9,10,"Actividades recreativas, deporte y cultura"
4,5,"Vivienda, agua, electricidad, gas y otros comb..."
6,7,Sanidad
0,1,Índice general
13,14,"Cuidado personal, protección social, y bienes ..."
8,9,Información y comunicaciones


In [34]:
df_sectores_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_sector      14 non-null     int64
 1   nombre_sector  14 non-null     str  
dtypes: int64(1), str(1)
memory usage: 356.0 bytes


In [35]:
df_sectores_ipc.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_sector,14.0,7.5,4.1833,1.0,4.25,7.5,10.75,14.0


In [36]:
df_sectores_ipc.describe(include='string').T

,count,unique,top,freq
nombre_sector,14,14,Índice general,1


In [37]:
df_sectores_ipc["nombre_sector"].unique()

<StringArray>
[                                                                    'Índice general',
                                                 'Alimentos y bebidas no alcohólicas',
                                                       'Bebidas alcohólicas y tabaco',
                                                                  'Vestido y calzado',
                             'Vivienda, agua, electricidad, gas y otros combustibles',
 'Muebles, artículos del hogar y artículos para el mantenimiento corriente del hogar',
                                                                            'Sanidad',
                                                                         'Transporte',
                                                       'Información y comunicaciones',
                                         'Actividades recreativas, deporte y cultura',
                                                                          'Enseñanza',
                             

------------------

In [38]:
df_territorio = pd.read_csv('../files/data_raw/territorio.csv')

In [39]:
df_territorio.sample(10)

,id_territorio,nombre_territorio
7,8,Castilla y León
11,12,Extremadura
16,17,País Vasco
17,18,"Rioja, La"
2,3,Aragón
0,1,Nacional
6,7,Cantabria
5,6,Canarias
12,13,Galicia
14,15,"Murcia, Región de"


In [40]:
df_territorio.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_territorio      20 non-null     int64
 1   nombre_territorio  20 non-null     str  
dtypes: int64(1), str(1)
memory usage: 452.0 bytes


In [41]:
df_territorio.describe(include='string').T

,count,unique,top,freq
nombre_territorio,20,20,Nacional,1


In [42]:
df_territorio["nombre_territorio"].unique()

<StringArray>
[                   'Nacional',                   'Andalucía',
                      'Aragón',     'Asturias, Principado de',
              'Balears, Illes',                    'Canarias',
                   'Cantabria',             'Castilla y León',
        'Castilla - La Mancha',                    'Cataluña',
        'Comunitat Valenciana',                 'Extremadura',
                     'Galicia',        'Madrid, Comunidad de',
           'Murcia, Región de', 'Navarra, Comunidad Foral de',
                  'País Vasco',                   'Rioja, La',
                       'Ceuta',                     'Melilla']
Length: 20, dtype: str

-----------------

In [43]:
df_tiempo = pd.read_csv('../files/data_raw/tiempo.csv')

In [44]:
df_tiempo.sample(10)

,id_tiempo,anio,mes,nombre_mes
59,202106,2021,6,Junio
227,200706,2007,6,Junio
237,200608,2006,8,Agosto
35,202306,2023,6,Junio
63,202102,2021,2,Febrero
58,202107,2021,7,Julio
0,202605,2026,5,Mayo
270,200311,2003,11,Noviembre
143,201406,2014,6,Junio
288,200205,2002,5,Mayo


In [45]:
df_tiempo.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_tiempo   294 non-null    int64
 1   anio        294 non-null    int64
 2   mes         294 non-null    int64
 3   nombre_mes  294 non-null    str  
dtypes: int64(3), str(1)
memory usage: 9.3 KB


In [46]:
df_tiempo.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,294.0,201381.948980,708.657083,200201.0,200802.25,201403.5,202004.75,202606.0
anio,294.0,2013.755102,7.087548,2002.0,2008.00,2014.0,2020.00,2026.0
mes,294.0,6.438776,3.457394,1.0,3.00,6.0,9.00,12.0


In [47]:
df_tiempo.duplicated().sum()

np.int64(0)

---------------------

In [48]:
df_tipo_medida = pd.read_csv('../files/data_raw/tipo_medida.csv')

In [49]:
df_tipo_medida.head(10)

,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [50]:
df_tipo_medida.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_medida      4 non-null      int64
 1   nombre_medida  4 non-null      str  
dtypes: int64(1), str(1)
memory usage: 196.0 bytes


In [51]:
df_tipo_medida.value_counts()

id_medida  nombre_medida                
1          Índice                           1
2          Variación mensual                1
3          Variación anual                  1
4          Variación en lo que va de año    1
Name: count, dtype: int64